In [3]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("Remove duplicate data"). \
getOrCreate()

In [9]:
people_list = [
    (1,"Thai", 26, "20M"),
    (1,"Thai", 26, "20M"),
    (1,"Thai", 26, "23M"),
    (1,"Thanh", 26, "30M"),
    (2,"Thanh", 26, "30M")
]

In [10]:
df = spark.createDataFrame(people_list).toDF("id", "name", "age", "salary")

In [11]:
df.show()

+---+-----+---+------+
| id| name|age|salary|
+---+-----+---+------+
|  1| Thai| 26|   20M|
|  1| Thai| 26|   20M|
|  1| Thai| 26|   23M|
|  1|Thanh| 26|   30M|
|  2|Thanh| 26|   30M|
+---+-----+---+------+



CÁCH 1: sử dụng distinct trong spark.sql

In [12]:
df1 = df.distinct()

In [13]:
df1.show()

+---+-----+---+------+
| id| name|age|salary|
+---+-----+---+------+
|  1| Thai| 26|   20M|
|  1| Thai| 26|   23M|
|  1|Thanh| 26|   30M|
|  2|Thanh| 26|   30M|
+---+-----+---+------+



In [14]:
df2 = df.dropDuplicates(["id"])

In [15]:
df2.show()

+---+-----+---+------+
| id| name|age|salary|
+---+-----+---+------+
|  1| Thai| 26|   20M|
|  2|Thanh| 26|   30M|
+---+-----+---+------+



In [17]:
#Duplicate 2 cot
df2_1 = df.dropDuplicates(["id", "salary"])

In [18]:
df2_1.show()

+---+-----+---+------+
| id| name|age|salary|
+---+-----+---+------+
|  1| Thai| 26|   20M|
|  1| Thai| 26|   23M|
|  1|Thanh| 26|   30M|
|  2|Thanh| 26|   30M|
+---+-----+---+------+



In [19]:
df2_2 = df.dropDuplicates(["id", "name"])

In [20]:
df2_2.show()

+---+-----+---+------+
| id| name|age|salary|
+---+-----+---+------+
|  1| Thai| 26|   20M|
|  1|Thanh| 26|   30M|
|  2|Thanh| 26|   30M|
+---+-----+---+------+



In [23]:
df3 = df.dropDuplicates()
# Giong nhu distinct

In [24]:
df3.show()

+---+-----+---+------+
| id| name|age|salary|
+---+-----+---+------+
|  1| Thai| 26|   20M|
|  1| Thai| 26|   23M|
|  1|Thanh| 26|   30M|
|  2|Thanh| 26|   30M|
+---+-----+---+------+



CÁCH 3: Không muốn saprk lựa chọn ngẫu nhiên, với vd này muốn lấy với salary lớn nhất

In [36]:
from pyspark.sql.window import *
from pyspark.sql.functions import row_number, desc

In [37]:
# window = Window.partitionBy('id').orderBy('salary')
window = Window.partitionBy('id').orderBy(desc('salary'))

In [38]:
df3 = df.withColumn("row_number", row_number().over(window))

In [39]:
df3.show()

+---+-----+---+------+----------+
| id| name|age|salary|row_number|
+---+-----+---+------+----------+
|  1|Thanh| 26|   30M|         1|
|  1| Thai| 26|   23M|         2|
|  1| Thai| 26|   20M|         3|
|  1| Thai| 26|   20M|         4|
|  2|Thanh| 26|   30M|         1|
+---+-----+---+------+----------+



In [40]:
df3 = df3.where("row_number = 1").drop("row_number")

In [41]:
df3.show()

+---+-----+---+------+
| id| name|age|salary|
+---+-----+---+------+
|  1|Thanh| 26|   30M|
|  2|Thanh| 26|   30M|
+---+-----+---+------+

